# Yapay Sinir Ağları (ANN) ile Şarap Kalitesi Sınıflandırma

Bu notebook, `yapay_sinir_aglari_egitimi.ipynb` tutorial'ında Iris veri seti üzerinde kurulan ANN mimarisinin, gerçek dünyadan gelen **daha gürültülü ve dengesiz bir veri seti** olan [Wine Quality (Red Wine) Dataset](https://www.kaggle.com/datasets/yasserh/wine-quality-dataset) üzerinde uygulanmasıdır.

Amaç aynı temel ANN yapısını (Dense + Dropout + Adam + one-hot encoding) korurken, gerçek veriyle çalışırken ortaya çıkan ek zorlukları (sınıf dengesizliği, kalite etiketlerinin gruplanması) ele almaktır.

**Veri seti:** 1143 örnek, 11 fizikokimyasal özellik (asitlik, şeker, alkol vb.) ve bir kalite puanı (3-8 arası, uzman tadımcılar tarafından verilmiş).


## 1. Gerekli Kütüphanelerin Yüklenmesi

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # TensorFlow bilgilendirme loglarını azalt

# Sayısal işlemler ve veri manipülasyonu
import numpy as np
import pandas as pd

# Görselleştirme
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: ön işleme ve metrikler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

# TensorFlow / Keras: model kurulumu
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

np.random.seed(42)
tf.random.set_seed(42)

print("Kütüphaneler başarıyla yüklendi.")


## 2. Wine Quality Veri Setinin Yüklenmesi

Veri setini [Kaggle - Wine Quality Dataset](https://www.kaggle.com/datasets/yasserh/wine-quality-dataset) sayfasından indirip `WineQT.csv` olarak bu notebook ile aynı klasöre koyun.


In [ ]:
df = pd.read_csv("WineQT.csv")

# Id sütunu modelleme için gereksiz, çıkarıyoruz
df = df.drop(columns=["Id"])

print(f"Veri seti boyutu: {df.shape[0]} örnek, {df.shape[1]} sütun")
df.head()


## 3. Keşifsel Veri Analizi (EDA)

Iris'in aksine bu veri seti dengesiz ve gerçek ölçüm gürültüsü içeriyor. Önce kalite dağılımına bakalım.

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x="quality", data=df, palette="viridis")
plt.title("Ham Kalite Puanı Dağılımı (3-8)")
plt.xlabel("Kalite Puanı")
plt.ylabel("Örnek Sayısı")
plt.show()

print(df["quality"].value_counts().sort_index())


**Gözlem:** Kalite puanları 3 ile 8 arasında değişiyor ancak uç değerler (3, 4, 8) çok az sayıda örneğe sahip. 6 sınıflı ham haliyle sınıflandırma yapmak, bazı sınıflarda train/test ayrımında yetersiz örnek kalmasına yol açar.

**Karar:** Kaliteyi 3 anlamlı gruba indirgeyeceğiz:
- **Düşük (0):** kalite ≤ 4
- **Orta (1):** kalite 5-6
- **Yüksek (2):** kalite ≥ 7

Bu, orijinal Iris tutorial'ındaki gibi 3 sınıflı bir yapı korurken, veriyi daha dengeli ve öğrenilebilir hale getirir.

In [ ]:
def bin_quality(q):
    if q <= 4:
        return 0  # Düşük
    elif q <= 6:
        return 1  # Orta
    else:
        return 2  # Yüksek

df["quality_group"] = df["quality"].apply(bin_quality)
label_names = {0: "Düşük", 1: "Orta", 2: "Yüksek"}

plt.figure(figsize=(6, 4))
sns.countplot(x="quality_group", data=df, palette="magma")
plt.xticks([0, 1, 2], ["Düşük", "Orta", "Yüksek"])
plt.title("Gruplanmış Kalite Dağılımı")
plt.show()

df["quality_group"].value_counts().rename(index=label_names)


Korelasyon ısı haritasıyla hangi özelliklerin kaliteyle daha ilişkili olduğuna bakalım.

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.drop(columns=["quality", "quality_group"]).assign(quality=df["quality"]).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Özellikler Arası Korelasyon Matrisi")
plt.show()


## 4. Verinin Ön İşlenmesi

Özellikleri (X) ve hedefi (y) ayırıyoruz, `StandardScaler` ile ölçekliyoruz ve eğitim/test setlerine bölüyoruz.

In [ ]:
X = df.drop(columns=["quality", "quality_group"]).values
y = df["quality_group"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Ölçekleme: sadece eğitim setine fit edilir, test setine sadece transform uygulanır (veri sızıntısını önlemek için)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Eğitim seti: {X_train.shape[0]} örnek")
print(f"Test seti:   {X_test.shape[0]} örnek")
print(f"Özellik sayısı: {X_train.shape[1]}")


## 5. Sınıf Dengesizliğiyle Başa Çıkma: Class Weight

"Düşük" sınıfı diğerlerine göre çok az örnek içeriyor. Model bu sınıfı görmezden gelmesin diye `class_weight` hesaplayıp eğitim sırasında kullanacağız.

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print("Sınıf ağırlıkları:", class_weight_dict)


## 6. Targeti One-Hot Encoding'e Çevirme

Categorical crossentropy loss fonksiyonu, çıkışın one-hot encoded formatta olmasını gerektirir.

In [ ]:
y_train_cat = to_categorical(y_train, num_classes=3)
y_test_cat = to_categorical(y_test, num_classes=3)

print("Örnek etiket:", y_train[0], "->", y_train_cat[0])


## 7. Yapay Sinir Ağı Modeli Oluşturma

Orijinal Iris tutorial'ındaki ile aynı mimari deseni kullanıyoruz: giriş katmanı → gizli katman (ReLU) → Dropout → ikinci gizli katman → softmax çıkış. Sadece giriş boyutu 11 özelliğe göre güncellendi.

In [ ]:
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(3, activation="softmax")
])

model.summary()


## 8. Modelin Derlenmesi (Compilation)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


## 9. Modelin Eğitilmesi

Dengesizliği telafi etmek için `class_weight` parametresini eğitime dahil ediyoruz.

In [ ]:
history = model.fit(
    X_train, y_train_cat,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    class_weight=class_weight_dict,
    verbose=1
)


## 10. Eğitim Sürecinin Görselleştirilmesi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["loss"], label="Eğitim Kaybı")
axes[0].plot(history.history["val_loss"], label="Doğrulama Kaybı")
axes[0].set_title("Kayıp (Loss) Eğrisi")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Eğitim Doğruluğu")
axes[1].plot(history.history["val_accuracy"], label="Doğrulama Doğruluğu")
axes[1].set_title("Doğruluk (Accuracy) Eğrisi")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


## 11. Test Verisi Üzerinde Değerlendirme

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test Kaybı: {test_loss:.4f}")
print(f"Test Doğruluğu: {test_accuracy:.4f}")


## 12. Tahminler ve Karmaşıklık Matrisi

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Düşük", "Orta", "Yüksek"],
            yticklabels=["Düşük", "Orta", "Yüksek"])
plt.xlabel("Tahmin Edilen")
plt.ylabel("Gerçek")
plt.title("Karmaşıklık Matrisi (Confusion Matrix)")
plt.show()


## 13. Detaylı Performans Raporu

In [ ]:
print(classification_report(
    y_test, y_pred,
    target_names=["Düşük", "Orta", "Yüksek"]
))


## 14. Özelliklerin Ağırlıklarının Görselleştirilmesi

İlk katmanın ağırlıklarının mutlak değer ortalaması, hangi özelliklerin modelin ilk katmanında daha güçlü etkiye sahip olduğuna dair kaba bir fikir verir.

In [ ]:
feature_names = df.drop(columns=["quality", "quality_group"]).columns
first_layer_weights = model.layers[0].get_weights()[0]  # shape: (n_features, 64)
importance = np.mean(np.abs(first_layer_weights), axis=1)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importance
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x="importance", y="feature", data=importance_df, palette="crest")
plt.title("İlk Katman Ağırlıklarına Göre Özellik Önemi (Yaklaşık)")
plt.show()


## 15. Yeni Veri ile Tahmin Yapma

Elle girilen fizikokimyasal ölçümlerle örnek bir tahmin yapalım. Sütun sırası eğitim verisiyle aynı olmalı.

In [ ]:
# fixed acidity, volatile acidity, citric acid, residual sugar, chlorides,
# free sulfur dioxide, total sulfur dioxide, density, pH, sulphates, alcohol
yeni_sarap = np.array([[7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]])
yeni_sarap_scaled = scaler.transform(yeni_sarap)

tahmin_probs = model.predict(yeni_sarap_scaled)
tahmin_sinif = np.argmax(tahmin_probs, axis=1)[0]

print(f"Tahmin edilen kalite grubu: {label_names[tahmin_sinif]}")
print(f"Sınıf olasılıkları -> Düşük: {tahmin_probs[0][0]:.3f}, Orta: {tahmin_probs[0][1]:.3f}, Yüksek: {tahmin_probs[0][2]:.3f}")


## 16. Özet ve Bulgular

| Konu | Açıklama |
|------|----------|
| **Veri seti** | Wine Quality (Kırmızı Şarap), 1143 örnek, 11 fizikokimyasal özellik |
| **Zorluk** | Ham kalite etiketleri (3-8) ciddi şekilde dengesiz; 3 gruba indirgenerek (Düşük/Orta/Yüksek) daha dengeli hale getirildi |
| **Mimari** | Iris tutorial'ıyla aynı desen: Dense(64) → Dropout → Dense(32) → Dropout → Softmax(3) |
| **Dengesizlik çözümü** | `class_weight="balanced"` ile azınlık sınıfının (Düşük) öğrenilmesi desteklendi |
| **Sonuç** | Model, "Orta" sınıfını (en kalabalık grup) en yüksek doğrulukla; "Düşük" sınıfını ise en zorlu şekilde tahmin ediyor — bu, gerçek dünya dengesiz verilerinde beklenen bir durumdur |

### Olası geliştirmeler
- Class weight yerine **SMOTE** gibi oversampling teknikleri denenebilir.
- Ham 6 sınıflı etiketle **regresyon** yaklaşımı (kaliteyi sürekli bir değer gibi tahmin etme) denenebilir.
- Farklı optimizatörler (SGD, RMSprop) ve öğrenme oranları karşılaştırılabilir.
